In [ ]:
%pip install scanpy leidenalg python-igraph scipy statsmodels gseapy

In [1]:
import sys, scanpy as sc
print(sys.version)
print(sc.__version__)

3.12.15 (main, Oct  9 2026, 15:21:55) [MSC v.1944 64 bit (AMD64)]
1.12.4


C:\Users\hp\AppData\Local\Temp\ipykernel_15064\3898917610.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(sc.__version__)


In [2]:
from pathlib import Path
import os

print(os.getcwd())
DATA = Path("../data")
f_tpm  = DATA / "GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz"
f_meta = DATA / "GSE120575_patient_ID_single_cells.txt.gz"
print(f_tpm.exists(), f_meta.exists())

f:\melanoma\melanoma-scrna-analysis\notebooks
True True


In [4]:
import gzip
with gzip.open(f_meta, "rt") as fh:
    for i, line in enumerate(fh):
        print(i, line[:150].rstrip())
        if i >= 30: break

0 # High-throughput sequencing metadata template (version 2.1).
1 # All fields in this template must be completed.
2 # Templates containing example data are found in the METADATA EXAMPLES spreadsheet tabs at the foot of this page.
3 # Field names (in blue on this page) should not be edited. Hover over cells containing field names to view field content guidelines.
4 "# Human data. If there are patient privacy concerns regarding making data fully public through GEO, please submit to NCBI's dbGaP (http://www.ncbi.nl
5 
6 SERIES
7 # This section describes the overall experiment.
8 title	Defining T cell states associated with response to checkpoint immunotherapy in melanoma
9 summary	"we profiled 16,291 immune cells from 48 tumor samples of melanoma patients treated with checkpoint inhibitors, using single-cell transcriptom
10 overall design	"Tumors were disaggregated, sorted into CD45+ single cells, and profiled by Smart-seq2"
11 contributor	Moshe Sade-Feldman
12 contributor	Keren Yizhak
1

In [5]:
with gzip.open(f_tpm, "rt") as fh:
    for i, line in enumerate(fh):
        print(i, len(line.split("\t")), line[:300].rstrip())
        if i >= 2: break

0 16292 	A10_P3_M11	A11_P1_M11	A11_P3_M11	A11_P4_M11	A12_P3_M11	A12_P6_M11	A2_P1_M11	A2_P4_M11	A3_P1_M11	A3_P3_M11	A4_P3_M11	A4_P4_M11	A4_P6_M11	A5_P4_M11	A5_P5_M11	A6_P1_M11	A6_P4_M11	A6_P6_M11	A7_P2_M11	A7_P6_M11	A8_P1_M11	A8_P6_M11	A9_P1_M11	A9_P4_M11	B10_P1_M11	B10_P3_M11	B10_P4_M11	B10_P6_M11	B11_P6_M1
1 16292 	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P1	Pre_P
2 16293 TSPAN6	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.00	0.0


In [6]:
with gzip.open(f_tpm, "rt") as fh:
    for i, line in enumerate(fh):
        parts = line.rstrip("\n").split("\t")
        print(i, "تعداد فیلد:", len(parts))
        print("   اولین فیلدها:", [repr(p) for p in parts[:3]])
        print("   آخرین فیلدها:", [repr(p) for p in parts[-3:]])
        if i >= 4:
            break

0 تعداد فیلد: 16292
   اولین فیلدها: ["''", "'A10_P3_M11'", "'A11_P1_M11'"]
   آخرین فیلدها: ["'H7_P5_M67_L001_T_enriched'", "'H8_P5_M67_L001_T_enriched'", "'H9_P5_M67_L001_T_enriched'"]
1 تعداد فیلد: 16292
   اولین فیلدها: ["''", "'Pre_P1'", "'Pre_P1'"]
   آخرین فیلدها: ["'Post_P6_T_enriched'", "'Post_P6_T_enriched'", "'Post_P6_T_enriched'"]
2 تعداد فیلد: 16293
   اولین فیلدها: ["'TSPAN6'", "'0.00'", "'0.00'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]
3 تعداد فیلد: 16293
   اولین فیلدها: ["'TNMD'", "'0.00'", "'0.00'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]
4 تعداد فیلد: 16293
   اولین فیلدها: ["'DPM1'", "'0.00'", "'9.24'"]
   آخرین فیلدها: ["'0.00'", "'0.00'", "''"]


In [8]:
import pandas as pd

meta = pd.read_csv(f_meta, sep="\t", skiprows=19, nrows=16291, encoding="latin-1")
cols = meta.columns.tolist()
print(cols)

meta = meta.rename(columns={
    cols[1]: "cell_id",
    cols[4]: "sample",
    cols[5]: "response",
    cols[6]: "therapy",
})

for c in ["cell_id", "sample", "response", "therapy"]:
    meta[c] = meta[c].str.strip()

meta = meta.set_index("cell_id")
meta["timepoint"]  = meta["sample"].str.split("_").str[0]
meta["patient"]    = meta["sample"].str.split("_").str[1]
meta["T_enriched"] = meta["sample"].str.contains("T_enriched")

print(meta.shape)
print(meta.index.is_unique)
print(meta["sample"].nunique(), meta["patient"].nunique())
print(meta["response"].value_counts())
print(meta["therapy"].value_counts())
print(meta.groupby("patient")["response"].nunique().max())
print(meta["T_enriched"].value_counts())
print(meta[["sample", "response", "therapy"]].tail(3))

['Sample name', 'title', 'source name', 'organism', 'characteristics: patinet ID (Pre=baseline; Post= on treatment)', 'characteristics: response', 'characteristics: therapy', 'molecule', 'description', 'processed data file ', 'raw file', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34']
(16291, 37)
True
48 32
response
Non-responder    10727
Responder         5564
Name: count, dtype: int64
therapy
anti-PD1          11653
anti-CTLA4+PD1     4121
anti-CTLA4          517
Name: count, dtype: int64
2
T_enriched
False    16291
Name: count, dtype: int64
                            sample       response   therapy
cell_id                                                    
H7_P5_M67_L001_T_e

In [10]:
meta = meta.loc[:, ~meta.columns.str.startswith("Unnamed")]

meta["T_enriched"] = meta.index.str.contains("T_enriched")
print(meta["T_enriched"].value_counts())

ct = pd.crosstab(meta["sample"], meta["T_enriched"])
print(ct[ct.get(True, 0) > 0])

n_lab = meta.groupby("patient")["response"].nunique()
mixed = n_lab[n_lab > 1].index.tolist()
print("بیماران با دو برچسب:", mixed)
print(meta[meta["patient"].isin(mixed)]
      .groupby(["patient", "sample", "response", "therapy"]).size())

T_enriched
False    15418
True       873
Name: count, dtype: int64
T_enriched  False  True 
sample                  
Post_P10      308     88
Post_P13      265    184
Post_P16      369     94
Post_P2       301     79
Post_P4       421    254
Post_P6       308     92
Post_P8       327     82
بیماران با دو برچسب: ['P1', 'P28', 'P4', 'P5']
patient  sample      response       therapy       
P1       Post_P1     Responder      anti-PD1          291
         Post_P1_2   Non-responder  anti-PD1          353
         Pre_P1      Responder      anti-CTLA4        229
P28      Post_P28    Non-responder  anti-CTLA4+PD1    369
         Post_P28_2  Non-responder  anti-CTLA4+PD1    372
         Pre_P28     Responder      anti-CTLA4+PD1    337
P4       Post_P4     Responder      anti-CTLA4+PD1    675
         Pre_P4      Non-responder  anti-CTLA4+PD1    311
P5       Post_P5     Non-responder  anti-PD1          292
         Post_P5_2   Responder      anti-PD1          336
dtype: int64


In [11]:
import pandas as pd

meta = pd.read_csv(f_meta, sep="\t", skiprows=19, nrows=16291, encoding="latin-1")
meta = meta.loc[:, ~meta.columns.str.startswith("Unnamed")]
cols = meta.columns.tolist()
print(cols)

meta = meta.rename(columns={
    cols[1]: "cell_id",
    cols[4]: "sample",
    cols[5]: "response",
    cols[6]: "therapy",
})

for c in ["cell_id", "sample", "response", "therapy"]:
    meta[c] = meta[c].str.strip()

meta = meta.set_index("cell_id")
meta["timepoint"]  = meta["sample"].str.split("_").str[0]
meta["patient"]    = meta["sample"].str.split("_").str[1]
meta["T_enriched"] = meta.index.str.contains("T_enriched")

print(meta.shape)
print(meta.index.is_unique)
print(meta["sample"].nunique(), meta["patient"].nunique())
print(meta["response"].value_counts())
print(meta["therapy"].value_counts())
print(meta.groupby("patient")["response"].nunique().max())
print(meta["T_enriched"].value_counts())
print(meta[["sample", "response", "therapy"]].tail(3))

['Sample name', 'title', 'source name', 'organism', 'characteristics: patinet ID (Pre=baseline; Post= on treatment)', 'characteristics: response', 'characteristics: therapy', 'molecule', 'description', 'processed data file ', 'raw file']
(16291, 13)
True
48 32
response
Non-responder    10727
Responder         5564
Name: count, dtype: int64
therapy
anti-PD1          11653
anti-CTLA4+PD1     4121
anti-CTLA4          517
Name: count, dtype: int64
2
T_enriched
False    15418
True       873
Name: count, dtype: int64
                            sample       response   therapy
cell_id                                                    
H7_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1
H8_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1
H9_P5_M67_L001_T_enriched  Post_P6  Non-responder  anti-PD1


In [12]:
extra = ["molecule", "description", "processed data file ", "raw file"]
print(meta[extra].nunique())
print(meta[extra].drop_duplicates().head())

molecule                0
description             0
processed data file     0
raw file                0
dtype: int64
            molecule  description  processed data file   raw file
cell_id                                                          
A10_P3_M11       NaN          NaN                   NaN       NaN


In [13]:
print(meta[["Sample name", "source name", "organism"]].nunique())

keep = ["sample", "response", "therapy", "timepoint", "patient", "T_enriched"]
meta = meta[keep].copy()

print(meta.shape)
print(meta.head(3))

Sample name    16291
source name        1
organism           1
dtype: int64
(16291, 6)
            sample   response     therapy timepoint patient  T_enriched
cell_id                                                                
A10_P3_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False
A11_P1_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False
A11_P3_M11  Pre_P1  Responder  anti-CTLA4       Pre      P1       False
